## No memory

In [3]:
%run langchain_common.py

In [4]:
agent = get_agent_instance(llm)

In [6]:
question = HumanMessage(content="Hello my name is Ali and my favourite colour is green")

response = agent.invoke(
    {"messages": [question]} 
)
pp(response["messages"][-1].content)

[
    {
        'summary': [
            {
                'text': 'Thinking Process:\n\n1.  **Analyze the Request:**\n    *   User\'s name: Ali\n    *   User\'s favorite color: green\n    *   Intent: Introduction, casual conversation starter.\n\n2.  **Determine the appropriate response:**\n    *   Acknowledge the user\'s name (Ali).\n    *   Acknowledge their favorite color (green).\n    *   Be friendly and welcoming.\n    *   Keep it concise but engaging.\n\n3.  **Drafting the response:**\n    *   *Option 1:* Hello Ali. Nice to meet you. Green is a nice color.\n    *   *Option 2:* Hi Ali! It\'s great to meet you. Green is a wonderful choice—it\'s fresh and calming. How can I help you today?\n    *   *Option 3:* Hello Ali! Nice to meet you. I like green too. What do you want to talk about?\n\n4.  **Refining the response (Option 2 is best):**\n    *   Greeting: "Hello Ali!"\n    *   Acknowledgment: "It\'s nice to meet you."\n    *   Comment on color: "Green is a fantastic color—it\'s o

In [8]:
question = HumanMessage(content="What's my favourite colour?")

response = agent.invoke(
    {"messages": [question]} 
)

pp(response["messages"][-1].content)

[
    {
        'summary': [
            {
                'text': 'Thinking Process:\n\n1.  **Analyze the Request:**\n    *   Question: "What\'s my favourite colour?"\n    *   Intent: The user is asking me to know something personal about them.\n    *   Constraint: I am an AI, I don\'t have access to the user\'s personal information unless they\'ve shared it with me in this conversation or if it\'s stored in a specific memory feature (which I don\'t have access to in this context).\n\n2.  **Determine Knowledge Base:**\n    *   I do not know the user.\n    *   I have no memory of past conversations (unless this is a continuous session with memory enabled, but even then, I should be cautious about assuming I know personal preferences without explicit confirmation).\n    *   Standard AI behavior: Acknowledge limitation, be friendly, invite the user to share.\n\n3.  **Formulate Response:**\n    *   Direct answer: I don\'t know.\n    *   Reason: I\'m an AI, I don\'t have access to personal

## Memory

In [10]:
# Checkpoints persist conversation state between turns so the agent can remember prior context.
# InMemorySaver is an in-process checkpoint backend: fast and simple for demos, but non-persistent across kernel restarts.

# Clarification:
# `messages` in agent.invoke(...) is only the per-call input payload (what you send this turn).
# Checkpoint memory is different: it stores and reloads the full conversation state across turns.
# That persisted state is keyed by `thread_id`, so memory works only when the same thread_id is reused.
# In short: message context is request-scoped; checkpoint memory is thread-scoped and persistent (in-process here).

from langgraph.checkpoint.memory import InMemorySaver  

agent = create_agent(
    llm,
    checkpointer=InMemorySaver(),  
)

In [11]:
from langchain.messages import HumanMessage

question = HumanMessage(content="Hello my name is Ali and my favourite colour is green")
config = {"configurable": {"thread_id": "1"}}

response = agent.invoke(
    {"messages": [question]},
    config,  
)

In [14]:
pp(response["messages"][-1].content)

[
    {
        'summary': [
            {
                'text': 'Thinking Process:\n\n1.  **Analyze the Request:**\n    *   User\'s name: Ali\n    *   User\'s favorite color: green\n    *   Intent: Introduction, casual conversation.\n\n2.  **Determine the appropriate response:**\n    *   Acknowledge the user\'s name (Ali).\n    *   Acknowledge their favorite color (green).\n    *   Be friendly and welcoming.\n    *   Keep it concise but engaging.\n\n3.  **Drafting the response:**\n    *   *Option 1:* Hello Ali. Nice to meet you. Green is a nice color.\n    *   *Option 2:* Hi Ali! It\'s nice to meet you. Green is a great choice for a favorite color. It\'s refreshing and natural.\n    *   *Option 3:* Hello Ali! Nice to meet you. I like green too. What do you like to do?\n\n4.  **Refining the response:**\n    *   Make it warm. "Hello Ali! It\'s nice to meet you."\n    *   Comment on the color. "Green is a wonderful color—very refreshing and calming."\n    *   Offer further assistance o

In [16]:
question = HumanMessage(content="What's my favourite colour?")

response = agent.invoke(
    {"messages": [question]},
    config,  
)

pp(response["messages"][-1].content)

[
    {
        'summary': [
            {
                'text': 'Thinking Process:\n\n1.  **Analyze the Request:** The user is asking "What\'s my favourite colour?" again.\n2.  **Recall Context:** In the very first message, the user (Ali) stated: "Hello my name is Ali and my favourite colour is green".\n3.  **Check Previous Interaction:** In the second turn, the user asked the same question, and I answered correctly ("green"). Now they are asking a third time.\n4.  **Determine the Answer:** The answer remains the same based on the initial information provided.\n5.  **Formulate Response:** State the color clearly (green) and perhaps add a friendly touch or emoji to keep the conversation engaging.\n6.  **Safety/Policy Check:** No sensitive or harmful content. Just a simple fact recall.\n7.  **Final Output:** "Your favourite colour is green!" (with maybe a green leaf emoji).\n\n*Self-Correction/Refinement:* Since I already answered this in the previous turn, I should just confirm it ag

In [18]:
# Demonstrate storing custom data in memory for the same thread
user_data = {
    "name": "Ali",
    "favorite_color": "green",
    "city": "Lahore",
    "loyalty_tier": "gold"
}

# Save custom data into the thread memory
save_message = HumanMessage(
    content=(
        "Store this custom data in memory and use it for future answers:\n"
        f"{user_data}"
    )
)
agent.invoke({"messages": [save_message]}, config)

# Ask a follow-up question that requires the stored custom data
question = HumanMessage(
    content="From my saved profile, what city do I live in and what is my loyalty tier?"
)
response = agent.invoke({"messages": [question]}, config)
pp(response["messages"][-1].content)

# Optional: inspect recent messages stored in memory for this thread
state = agent.get_state(config)
pp(state.values["messages"][-4:])

[
    {
        'summary': [
            {
                'text': 'Thinking Process:\n\n1.  **Analyze the Request:** The user is asking for specific information from their "saved profile" (city and loyalty tier). This is a follow-up to previous interactions where they provided this data.\n\n2.  **Recall Context:**\n    *   User introduced themselves as Ali.\n    *   User stated their favorite color is green.\n    *   User explicitly provided a JSON-like structure: `{\'name\': \'Ali\', \'favorite_color\': \'green\', \'city\': \'Lahore\', \'loyalty_tier\': \'gold\'}`.\n    *   I (the AI) acknowledged this data in previous turns.\n    *   The user is now testing if I remember/retrieve this information.\n\n3.  **Check Constraints & Capabilities:**\n    *   I am an AI language model. I don\'t have persistent memory across sessions unless the platform provides a specific memory feature (which seems to be simulated here based on the conversation history).\n    *   Within *this* conversation 

In [19]:
def ask(thread_config, text):
    result = agent.invoke({"messages": [HumanMessage(content=text)]}, thread_config)
    return result["messages"][-1].content

In [22]:

# Scenario 1: Thread isolation (different users/sessions)
config_a = {"configurable": {"thread_id": "memory-demo-A"}}
config_b = {"configurable": {"thread_id": "memory-demo-B"}}

ask(config_a, "My preferred database is PostgreSQL.")
ask(config_b, "My preferred database is MongoDB.")

pp("Scenario 1A:", ask(config_a, "What database do I prefer?"))
pp("Scenario 1B:", ask(config_b, "What database do I prefer?"))

AttributeError: 'list' object has no attribute 'write'

In [21]:
# Scenario 2: Preference update within the same thread
config_update = {"configurable": {"thread_id": "memory-demo-update"}}

ask(config_update, "My deployment region is East US.")
ask(config_update, "Actually, update that: my deployment region is West Europe.")
pp("Scenario 2:", ask(config_update, "Which deployment region should you use now?"))

# Scenario 3: Multi-step task memory
config_task = {"configurable": {"thread_id": "memory-demo-task"}}

ask(config_task, "I'm building a chatbot MVP. Deadline is Friday. Budget is $2,000.")
pp("Scenario 3A:", ask(config_task, "Summarize my constraints in one sentence."))
pp("Scenario 3B:", ask(config_task, "Given those constraints, suggest 3 practical next steps."))

# Optional: inspect latest stored messages for one scenario
task_state = agent.get_state(config_task)
pp("\nRecent messages in Scenario 3 thread:")
pp(task_state.values["messages"][-4:])

Scenario 2: [{'type': 'reasoning', 'summary': [{'type': 'summary_text', 'text': 'Thinking Process:\n\n1.  **Analyze the Request:** The user is asking "Which deployment region should you use now?" based on the previous conversation.\n2.  **Review Conversation History:**\n    *   User initially said: "My deployment region is East US."\n    *   Assistant acknowledged East US.\n    *   User corrected: "Actually, update that: my deployment region is West Europe."\n    *   Assistant acknowledged West Europe.\n3.  **Determine the Current State:** The most recent instruction from the user is that their deployment region is **West Europe**.\n4.  **Formulate the Answer:** The assistant should confirm that the current deployment region to use is West Europe, as per the user\'s latest update.\n5.  **Refine the Answer:** Keep it clear and direct. Acknowledge the update and confirm the region.\n\n*Self-Correction/Edge Case Consideration:* Is there any ambiguity? The user asked "Which deployment regi